# Camera Calibration and Evaluation

This notebook follows the same flow as the Lab 2 handout.

1. Load the detected board corners.
2. Check that the board description matches the printed board you measured in class.
3. Run calibration and save the intrinsics.
4. Evaluate the fit with a few simple visual checks.
5. Use the saved `intrinsics.yml` in `measure_object.py`.

Run the corner detector first so that `captured_points/corners.npz` exists.


In [ ]:
from pathlib import Path
import sys

import cv2
import matplotlib.pyplot as plt
import numpy as np
import yaml

if not hasattr(cv2, 'aruco'):
    raise ImportError('cv2.aruco is not available. Install opencv-contrib-python on the laptop.')

cwd = Path.cwd().resolve()
if (cwd / 'board_config.py').exists():
    src_dir = cwd
elif (cwd / 'src' / 'board_config.py').exists():
    src_dir = cwd / 'src'
else:
    raise RuntimeError('Open this notebook from Lab_2/src or Lab_2.')

if str(src_dir) not in sys.path:
    sys.path.insert(0, str(src_dir))

from board_config import (
    CAPTURED_POINTS_DIR,
    CHARUCO_MARKER_SIZE_M,
    CHARUCO_SQUARES_X,
    CHARUCO_SQUARES_Y,
    CHECKERBOARD_INNER_CORNERS,
    SQUARE_SIZE_M,
    charuco_object_points_for_ids,
    create_charuco_board,
)

np.set_printoptions(suppress=True, precision=4)
plt.style.use('default')

npz_path = CAPTURED_POINTS_DIR / 'corners.npz'
yaml_out = CAPTURED_POINTS_DIR / 'intrinsics.yml'

if not npz_path.exists():
    raise FileNotFoundError(f'Could not find {npz_path}. Run detect_corners.py first.')

print(f'Corners file: {npz_path}')
print(f'Intrinsics output: {yaml_out}')


In [ ]:
data = np.load(npz_path, allow_pickle=False)
mode = 'checker' if 'objpoints' in data.files else 'charuco'

image_size = tuple(int(v) for v in data['img_shape'])
valid_files = [Path(p) for p in data['valid_files'].tolist()]

print(f'Calibration mode: {mode}')
print(f'Image size: {image_size[0]} x {image_size[1]}')
print(f'Valid frames kept by the detector: {len(valid_files)}')

if mode == 'checker':
    objpoints = [frame.astype(np.float32) for frame in data['objpoints']]
    imgpoints = [frame.astype(np.float32) for frame in data['imgpoints']]
    expected = CHECKERBOARD_INNER_CORNERS[0] * CHECKERBOARD_INNER_CORNERS[1]
    print(f'Valid frame rule: all {expected} checkerboard inner corners were found.')
else:
    board = create_charuco_board()
    corners = [frame.astype(np.float32) for frame in data['corners']]
    ids = [frame.astype(np.int32) for frame in data['ids']]
    expected = (CHARUCO_SQUARES_X - 1) * (CHARUCO_SQUARES_Y - 1)
    print(f'Valid frame rule: the current detector keeps frames with all {expected} ChArUco corners.')


## Step 1. Confirm the board configuration

Before you calibrate, check that these numbers match the printed board you measured in class.

If the square size or marker size is wrong, the scale of the calibration and the later measurement step will also be wrong.


In [ ]:
print('Checkerboard inner corners:', CHECKERBOARD_INNER_CORNERS)
print(f'Checkerboard square size: {SQUARE_SIZE_M * 100:.2f} cm')
print(f'ChArUco squares: {CHARUCO_SQUARES_X} x {CHARUCO_SQUARES_Y}')
print(f'ChArUco square size: {SQUARE_SIZE_M * 100:.2f} cm')
print(f'ChArUco marker size: {CHARUCO_MARKER_SIZE_M * 100:.2f} cm')
print('\nIf these do not match your printed board, update src/board_config.py and rerun the relevant steps.')


## Step 2. Run calibration

The calibration solves for two things at the same time.

- The camera intrinsics and distortion that stay fixed for this camera.
- The pose of the board in each frame.

The mean reprojection error is useful, but it is not enough on its own. We will inspect the fit more carefully in the next steps.


In [ ]:
if mode == 'checker':
    ret, K, d, rvecs, tvecs = cv2.calibrateCamera(
        objpoints,
        imgpoints,
        image_size,
        None,
        None,
    )
else:
    ret, K, d, rvecs, tvecs = cv2.aruco.calibrateCameraCharuco(
        charucoCorners=corners,
        charucoIds=ids,
        board=board,
        imageSize=image_size,
        cameraMatrix=None,
        distCoeffs=None,
    )

payload = {
    'K': {'data': K.tolist(), 'rows': 3, 'cols': 3},
    'D': {'data': d.ravel().tolist(), 'rows': 1, 'cols': int(d.size)},
}
with open(yaml_out, 'w', encoding='utf-8') as f:
    yaml.safe_dump(payload, f, sort_keys=False)

print(f'Mean reprojection error from OpenCV: {ret:.4f} px')
print('K =\n', K)
print('d =\n', d.ravel())
print(f'Saved intrinsics to {yaml_out}')


## Step 3. Evaluate the calibration

Use the next cells to answer a few questions.

1. Is the overall fit good, or are the errors clearly too large?
2. Do a few frames look much worse than the rest?
3. Does the model struggle more near the image edges than near the center?
4. Do the reprojected corners land back on the detected corners?
5. Does undistortion change the image in a plausible way?


In [ ]:
frame_reports = []
all_errors = []
all_radial = []
all_xy = []

for idx, file_path in enumerate(valid_files):
    if mode == 'checker':
        objp = objpoints[idx]
        detected = imgpoints[idx].reshape(-1, 2)
    else:
        objp = charuco_object_points_for_ids(board, ids[idx])
        detected = corners[idx].reshape(-1, 2)

    projected, _ = cv2.projectPoints(objp, rvecs[idx], tvecs[idx], K, d)
    projected = projected.reshape(-1, 2)

    point_errors = np.linalg.norm(projected - detected, axis=1)
    radii = np.linalg.norm(detected - K[:2, 2], axis=1)

    all_errors.extend(point_errors.tolist())
    all_radial.extend(radii.tolist())
    all_xy.append(detected)

    frame_reports.append(
        {
            'index': idx,
            'file': file_path.name,
            'mean_error': float(point_errors.mean()),
            'max_error': float(point_errors.max()),
            'points': int(len(point_errors)),
        }
    )

all_errors = np.asarray(all_errors, dtype=np.float32)
all_radial = np.asarray(all_radial, dtype=np.float32)
all_xy = np.vstack(all_xy)

print(f'Total detected points used in calibration: {len(all_errors)}')
print(f'Mean point error: {all_errors.mean():.4f} px')
print(f'Median point error: {np.median(all_errors):.4f} px')
print(f'95th percentile point error: {np.percentile(all_errors, 95):.4f} px')

worst_frames = sorted(frame_reports, key=lambda item: item['mean_error'], reverse=True)[:5]
print('\nWorst frames by mean reprojection error:')
for item in worst_frames:
    print(
        f"{item['index']:>2}  {item['file']:<20}  mean={item['mean_error']:.4f}px  max={item['max_error']:.4f}px  points={item['points']}"
    )

fig, axes = plt.subplots(1, 2, figsize=(12, 4))

upper = max(np.percentile(all_errors, 99), 0.5)
axes[0].hist(all_errors, bins=30, range=(0, upper), color='0.25')
axes[0].set_title('Reprojection error distribution')
axes[0].set_xlabel('Error in pixels')
axes[0].set_ylabel('Count')

axes[1].scatter(all_radial, all_errors, s=8, alpha=0.35, color='tab:blue')
axes[1].set_title('Error versus distance from the principal point')
axes[1].set_xlabel('Radius in pixels')
axes[1].set_ylabel('Error in pixels')

plt.tight_layout()
plt.show()


In [ ]:
example_idx = len(valid_files) // 2
example_path = valid_files[example_idx]
example_bgr = cv2.imread(str(example_path))

if mode == 'checker':
    example_objp = objpoints[example_idx]
    example_detected = imgpoints[example_idx].reshape(-1, 2)
else:
    example_objp = charuco_object_points_for_ids(board, ids[example_idx])
    example_detected = corners[example_idx].reshape(-1, 2)

example_projected, _ = cv2.projectPoints(
    example_objp,
    rvecs[example_idx],
    tvecs[example_idx],
    K,
    d,
)
example_projected = example_projected.reshape(-1, 2)

overlay = example_bgr.copy()
for x, y in example_detected:
    cv2.circle(overlay, (int(round(x)), int(round(y))), 5, (0, 255, 0), 1)
for u, v in example_projected:
    cv2.circle(overlay, (int(round(u)), int(round(v))), 3, (0, 0, 255), -1)

undistorted = cv2.undistort(example_bgr, K, d)

fig, axes = plt.subplots(1, 2, figsize=(12, 5))
axes[0].imshow(cv2.cvtColor(overlay, cv2.COLOR_BGR2RGB))
axes[0].set_title(f'Green detected, red reprojected\n{example_path.name}')
axes[0].axis('off')

axes[1].imshow(cv2.cvtColor(undistorted, cv2.COLOR_BGR2RGB))
axes[1].set_title('Undistorted image')
axes[1].axis('off')

plt.tight_layout()
plt.show()


In [ ]:
grid_x = np.linspace(0, image_size[0] - 1, 18)
grid_y = np.linspace(0, image_size[1] - 1, 12)
gx, gy = np.meshgrid(grid_x, grid_y)
grid_points = np.stack([gx, gy], axis=-1).astype(np.float32).reshape(-1, 1, 2)

undistorted_grid = cv2.undistortPoints(grid_points, K, d, P=K).reshape(-1, 2)
original_grid = grid_points.reshape(-1, 2)
shift = undistorted_grid - original_grid

fig, axes = plt.subplots(1, 2, figsize=(12, 5))

axes[0].quiver(
    original_grid[:, 0],
    original_grid[:, 1],
    shift[:, 0],
    shift[:, 1],
    angles='xy',
    scale_units='xy',
    scale=1,
    color='tab:red',
)
axes[0].set_xlim(0, image_size[0])
axes[0].set_ylim(image_size[1], 0)
axes[0].set_aspect('equal')
axes[0].set_title('Undistortion vector field')
axes[0].set_xlabel('u')
axes[0].set_ylabel('v')

hb = axes[1].hexbin(
    all_xy[:, 0],
    all_xy[:, 1],
    C=all_errors,
    reduce_C_function=np.mean,
    gridsize=18,
    cmap='magma',
)
axes[1].set_xlim(0, image_size[0])
axes[1].set_ylim(image_size[1], 0)
axes[1].set_title('Mean error by image location')
axes[1].set_xlabel('u')
axes[1].set_ylabel('v')
cb = fig.colorbar(hb, ax=axes[1])
cb.set_label('Mean error in pixels')

plt.tight_layout()
plt.show()


## Next step

The saved `intrinsics.yml` can now be used with `measure_object.py`.

That measurement step only works when the clicked object points lie on the same physical plane as the calibration board. In other words, the method uses the board plane constraint `Z = 0` to turn image points back into real-world distances on that plane.

If you have time, repeat the full workflow with the other board type and compare the calibration quality, the number of usable frames, and the clarity of the evaluation plots.
